***DinoV2 + SuperPoint + LightGlue*** UAV-VisLoc

# Preparations

## Installing requirements

In [3]:

!pip install -q faiss-gpu gdown \
    "lightglue @ git+https://github.com/cvg/LightGlue@eb42fee2d71449efb0aa5c10549752b5d75384d8"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 127.0 MB/s eta 0:00:00


## Imports

In [4]:
import cv2
import csv
import torch
import pandas as pd
import glob
import time
import os
import gc
import numpy as np
import pickle
import faiss
import random
import rasterio
import torch.nn.functional as F
import torchvision.transforms as T

from tqdm import tqdm
from matplotlib import pyplot as plt
from matplotlib.patches import Polygon
from torch.utils.data import Dataset, DataLoader, random_split
from rasterio.windows import Window
from lightglue import SuperPoint, LightGlue
from lightglue.utils import numpy_image_to_torch, rbd
from transformers import AutoModel
from typing import Tuple, Dict, List, Any

In [5]:
print(os.getcwd())

/content


## Downloading the dataset

In [6]:
region_id = '06'

regions_list = {
    '06' : '1deCVWgPquc6TCj0m9RFu1STbxHE9cehW'
}

In [7]:
import os
import zipfile
import gdown

if not os.path.exists('/content/dataset'):
    os.makedirs('/content/dataset')

zip_path = "./dataset/dataset.zip"
extract_path = "./dataset"

# url = f"https://drive.google.com/uc?id={region}"
print("Завантаження zip-архіву...")
gdown.download(id=regions_list[region_id], output=zip_path, quiet=False)

print("Розпакування архіву...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

if os.path.exists(zip_path):
    os.remove(zip_path)

print("Готово! Файли успішно розміщено в локальній пам'яті.")

Завантаження zip-архіву...


Downloading...
From (original): https://drive.google.com/uc?id=1deCVWgPquc6TCj0m9RFu1STbxHE9cehW
From (redirected): https://drive.google.com/uc?id=1deCVWgPquc6TCj0m9RFu1STbxHE9cehW&confirm=t&uuid=3cb5878d-1f14-4790-a410-85a72d323867
To: /content/dataset/dataset.zip
100%|██████████| 694M/694M [00:08<00:00, 80.4MB/s] 


Розпакування архіву...
Готово! Файли успішно розміщено в локальній пам'яті.


In [8]:
print(os.getcwd())

/content


# Preprocessing

In [9]:
# config
from omegaconf import OmegaConf
from pathlib import Path

base_path = Path(str(os.getcwd()) + "/dataset/06")
# hf_url = Path('https://huggingface.co/datasets/mvidem/UAV-VisLoc')
config = {
    "device" : "cuda",
    "model": {
        "top_k": 20,
        "num_keypoints": 2048,
        'magsac_thresh' : 5.0,
    },
    "path": {
        "meta" : str(base_path / "06.csv"),
        "metadata": str(base_path / "metadata"),
        "faiss": str(base_path / "metadata"),
        "satellite": str(base_path / "satellite06.tif"),
        "images_dir": str(base_path / "drone"),
    },
    "tiles" : {
        "batch_size" : 32,
        "stride" : 256,
        "tile_size" : 1024,
        "dino_size" : 448,
        "embed_dim" : 768,
        'shuffle' : False,
        'num_workers' : 2,
    },
    'fallback' : {
        'top_k' : 5,
        'temp' : 0.05
    },
    'rotation_retry_ranks' : 3,
    'inliers' : {
        'good' : 150,
        'min' : 15,
        'rotation_retry_min' : 8,
    },
    "dataset" : {
        "batch_size" : 16,
        "shuffle" : False,
        "val_split" : None,
        "num_workers" : 2,
        'scales' : (0.75, 1.0, 1.25),
        'n_rotations' : 4,
        'agg' : 'gem',
    },
    "seed" : 42,
    "gsd_drone" : 0.11,  # м/px; оцінено з геометрії камери та метаданих
    "gsd_map" : 0.27509135588521216 # оцінено з геометрії камери та метаданих
}
config = OmegaConf.create(config)

In [10]:
if config.device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Cuda is not available. Project is heavy and requires cuda!")
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [11]:
dinov2_model = AutoModel.from_pretrained("facebook/dinov2-giant", dtype=torch.float16) \
        .to(device).eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

In [12]:
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

# Processing

## Initialization of utils

In [13]:
def seed_all(seed: int = 143):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    cv2.setRNGSeed(seed)  # MAGSAC стохастичний

seed_all(config.seed)

In [14]:
class Maps():
    def __init__(self, current_map : str = '06'):
        self.current_map = current_map

    def get_current_map(self):
        map_path = f'{os.getcwd()}/dataset/{self.current_map}/satellite{self.current_map}.tif'
        return self.current_map, map_path

    def set_current_map(self, region_id : str):
        if not (os.path.exists(f'{os.getcwd()}/dataset/{self.region_id}/satellite{self.region_id}.tif') or \
                os.path.exists(f'{os.getcwd()}/dataset/{self.region_id}/drone') or \
                os.path.exists(f'{os.getcwd()}/dataset/{self.region_id}/{self.region_id}.csv')):
            raise RuntimeError(f"Can't change current working dir to {region_id}, becouse it's empty!")
        self.current_map = region_id

maps_list = Maps(current_map=region_id)

In [15]:
def get_map():
    global maps_list
    if maps_list is None:
        raise RuntimeError("Class for processing Satellete maps is not initialized!")
    return maps_list.get_current_map()


In [16]:
_, path = get_map()
print(path)

/content/dataset/06/satellite06.tif


In [17]:
def prepare_map_data() -> Tuple[Dict[str, float], int, int]:
    map_id, map_path = get_map()

    if not hasattr(prepare_map_data, "maps_data"):
        prepare_map_data.maps_data = {}

    if map_id not in prepare_map_data.maps_data:

        with rasterio.open(map_path) as m:
            b = m.bounds
            bounds = {
                'max_lat' : b.top,
                'min_lat' : b.bottom,
                'max_lon' : b.right,
                'min_lon' : b.left
            }
            map_width = m.width
            map_height = m.height
        prepare_map_data.maps_data[map_id] = {'bounds' : bounds, 'map_width' : map_width, 'map_height' : map_height}

    map_data = prepare_map_data.maps_data[map_id]
    return map_data['bounds'], map_data['map_width'], map_data['map_height']

In [18]:
def latlon_to_xy(
        lat : float | np.ndarray,
        lon : float | np.ndarray,
) -> tuple:
    bounds, map_width, map_height = prepare_map_data()
    x = (np.asarray(lon) - bounds['min_lon']) / (bounds['max_lon'] - bounds['min_lon']) * map_width
    y = (bounds['max_lat'] - np.asarray(lat)) / (bounds['max_lat'] - bounds['min_lat']) * map_height

    return x, y

def xy_to_latlon(
        x : float | np.ndarray,
        y: float | np.ndarray,
) -> tuple:
    bounds, map_width, map_height = prepare_map_data()
    lon = np.asarray(x) / map_width * (bounds['max_lon'] - bounds['min_lon']) + bounds['min_lon']
    lat = bounds['max_lat'] - np.asarray(y) / map_height * (bounds['max_lat'] - bounds['min_lat'])

    return lat, lon

In [19]:
def mean_haversine_error(
    lat1: float | np.ndarray,
    lon1: float | np.ndarray,
    lat2: float | np.ndarray,
    lon2: float | np.ndarray,
    r_earth: float = 6_371_000.0,
) -> float | np.ndarray:
    """Геодезична відстань між точками на сфері (метри), формула haversine з умови.

    Основна метрика похибки локалізації (MHE): відстань між передбаченою та GT-координатою.

    Args:
        lat1, lon1: широта/довгота першої точки (число або масив).
        lat2, lon2: широта/довгота другої точки (число або масив).
        r_earth: радіус Землі в метрах (за умовою — 6371000).

    Returns:
        відстань у метрах (число або масив тієї ж форми, що й вхід).
    """
    p1, p2 = (
        np.radians(np.asarray(lat1)),
        np.radians(np.asarray(lat2)),
    )
    dl = np.radians(np.asarray(lon2) - np.asarray(lon1))
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * r_earth * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

In [20]:
def to_square(image: np.ndarray | torch.Tensor) -> np.ndarray:
    if isinstance(image, torch.Tensor):
        image = image.numpy()
    h, w = image.shape[:2]
    if h == w:
        return image
    size = max(h, w)
    return cv2.resize(image, (size, size), interpolation = cv2.INTER_AREA)

In [21]:
def rotate_image(
        image : np.ndarray,
        angle : float
) -> np.ndarray:
    h, w = image.shape[:2]
    matrix = cv2.getRotationMatrix2D(center=(w/2, h/2), angle=angle, scale=1.0)
    return cv2.warpAffine(image, matrix, (w, h))

In [22]:
def footprint_corners(
        cx : float,
        cy : float,
        w : float,
        h : float,
        angle : float
) -> np.ndarray:
    angle_rad = np.radians(angle)
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    corners = np.array([
        [-w/2, -h/2],
        [w/2, -h/2],
        [w/2, h/2],
        [-w/2, h/2]
    ])
    rotation = np.array([[c, -s], [s, c]])
    return corners @ rotation.T + [cx, cy]

In [23]:
def read_map_window(x0 : int, y0 : int, size : int) -> np.ndarray:
    _, map_path = get_map()

    with rasterio.open(map_path) as src:
        x0c, y0c = max(0, x0), max(0, y0)
        w = min(size, src.width - x0c)
        h = min(size, src.height - y0c)
        arr = src.read(indexes=[1,2,3], window=Window(x0c, y0c, w, h))
    return np.ascontiguousarray(arr.transpose(1,2,0))

## Datasets

In [24]:
class UAVDataset(Dataset):
    def __init__(self, images_dir, satellite, metadata, transform : T.Compose = None):
        super().__init__()

        self.images_dir = images_dir
        self.satellite = cv2.imread(satellite)
        self.metadata = pd.read_csv(metadata)
        self.transform = transform

        images_path = glob.glob(os.path.join(images_dir, "*.JPG"))

        self.images = sorted(os.path.basename(p) for p in images_path)

    def __getitem__(self, idx):
        image_path = os.path.join(self.images_dir, self.images[idx])
        image_bgr = cv2.imread(image_path)

        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        meta = self.metadata.iloc[idx].to_dict()

        if self.transform is not None:
            image_rgb = self.transform(image_rgb)
            # image_bgr = self.transform(image_bgr)

        return image_rgb, meta, image_bgr

    def __len__(self):
        return len(self.images)

    def get_random_image(self):
        if len(self.images) == 0:
            raise ValueError("Cannot select an image because the dataset is empty.")

        idx = random.randrange(len(self.images))
        return self[idx]

    def get_random_image_data(self):
        if len(self.images) == 0:
            raise ValueError("Cannot select an image because the dataset is empty.")

        idx = random.randrange(len(self.images))

        image_rgb, meta, _ = self[idx]
        image_path = os.path.join(self.images_dir, self.images[idx])
        return {
            "idx" : idx,
            "image_rgb" : image_rgb,
            "path" : image_path,
            "meta" : meta,
        }
    def find_by_name(self, filename: str):
        """
        Пошук фото за назвою (без шляху).
        Повертає дані так само, як __getitem__.
        """
        if filename not in self.images:
            raise ValueError(f"Image '{filename}' not found in dataset.")
        idx = self.images.index(filename)
        return self[idx]

images_dataset = UAVDataset(
    config.path.images_dir, config.path.satellite, config.path.meta
)


In [25]:
# image, meta, _ = images_dataset.find_by_name("06_0001.JPG")
#
# plt.imshow(image)
#
# print(meta['filename'])

In [26]:
_, map_width, map_height = prepare_map_data()
n_show = 6
idxs_show = np.linspace(0, len(images_dataset) - 1, n_show, dtype=int)

fig, axs = plt.subplots(n_show, 2, figsize=(9, 3 * n_show))
for row, idx in enumerate(idxs_show):
    image_rgb, meta, image_brg = images_dataset[idx]
    gt_x, gt_y = latlon_to_xy(lat=meta['lat'], lon=meta['lon'])
    fp = int(image_rgb.shape[1] * config.gsd_drone / config.gsd_map)  # footprint по ширині кадру
    # Вікно центрується на GT, але зсувається (не просто обрізається), щоб лишитись
    # у межах карти — інакше GT біля краю карти виявиться в кутку вікна, а не по центру.
    win_x0 = max(0, min(int(gt_x) - fp // 2, map_width - fp))
    win_y0 = max(0, min(int(gt_y) - fp // 2, map_height - fp))
    win = read_map_window(win_x0, win_y0, fp)
    axs[row, 0].imshow(image_rgb)  # кадр як є (без кропа, без повороту)
    axs[row, 0].set_title(f'image #{meta['filename']}', fontsize=8)
    axs[row, 0].axis("off")
    axs[row, 1].imshow(win)
    axs[row, 1].set_title("карта навколо GT", fontsize=8)
    axs[row, 1].axis("off")
plt.tight_layout()
plt.show()

In [27]:
n_show = 6
idxs_show = np.linspace(0, len(images_dataset) - 1, n_show, dtype=int)

fig, axs = plt.subplots(n_show, 2, figsize=(11, 4 * n_show))
for i, idx in enumerate(idxs_show):
    image_rgb, meta, _ = images_dataset[idx]
    gt_x, gt_y = latlon_to_xy(lat=meta['lat'], lon=meta['lon'])
    w_px = image_rgb.shape[1] * config.gsd_drone / config.gsd_map
    h_px = image_rgb.shape[0] * config.gsd_drone / config.gsd_map
    size = int(max(w_px, h_px) * 2.5)
    x0 = max(0, min(int(gt_x) - size // 2, map_width - size))
    y0 = max(0, min(int(gt_y) - size // 2, map_height - size))
    win = read_map_window(x0, y0, size)
    x0c, y0c = (
        x0,
        y0,
    )

    axs[i, 0].imshow(rotate_image(image_rgb, -meta['Phi1']))
    axs[i, 0].set_title(f"image ({meta['filename']}) ↻ на північ", fontsize=8)
    axs[i, 0].axis("off")

    axs[i, 1].imshow(win)
    axs[i, 1].plot(gt_x - x0c, gt_y - y0c, "r+", ms=14, mew=2)
    box = footprint_corners(gt_x - x0c, gt_y - y0c, w_px, h_px, meta['Phi1'])
    axs[i, 1].add_patch(Polygon(box, fill=False, edgecolor="yellow", lw=1.5))
    axs[i, 1].set_title("карта 2.5x: + = GT, рамка = footprint", fontsize=8)
    axs[i, 1].axis("off")

    print(
        f"{meta['filename']}: height={meta['height']}м, w_px={w_px:.0f}, h_px={h_px:.0f}, size={size}, map={map_width}x{map_height}"
    )
    print(
        f"{meta['filename']}: gt_x/map_width={gt_x / map_width:.2f}, gt_y/map_height={gt_y / map_height:.2f}"
    )

plt.tight_layout()
plt.show()


In [28]:
def build_images_loader(dataset: UAVDataset):
    common = dict(
        batch_size=config.dataset.batch_size,
        shuffle=config.dataset.shuffle,
        num_workers=config.dataset.num_workers,
        pin_memory=True if device.type == "cuda" else False,
    )
    return DataLoader(dataset, **common)


In [29]:
class TilesDataset(Dataset):
    def __init__(self, tile_size : int, stride : int) -> List[Tuple[int, int]]:
        super().__init__()
        self.tile_size = tile_size
        self.stride = stride

        self.tiles : List[Tuple[int, int]] = self.cut_map_tiles()

    def __call__(self) -> List[Tuple[int, int]]:
        return self.tiles

    def __len__(self):
        return len(self.tiles)

    def cut_map_tiles(self):
        _, map_width, map_height = prepare_map_data()

        xs = [x for x in range(1, max(1, map_width - self.tile_size + 1), self.stride)]
        ys = [y for y in range(1, max(1, map_height - self.tile_size + 1), self.stride)]

        if xs[-1] + self.tile_size < map_width:
            xs.append(map_width - self.tile_size)
        if ys[-1] + self.tile_size < map_height:
            ys.append(map_height - self.tile_size)

        return [(x, y) for y in ys for x in xs]

    def __getitem__(self, idx):
        return self.tiles[idx]


In [30]:
def build_tiles_loader(dataset : TilesDataset) -> DataLoader:
    common = dict(
        batch_size=config.tiles.batch_size,
        shuffle=config.tiles.shuffle,
        num_workers=config.tiles.num_workers,
        pin_memory=True if device.type == "cuda" else False,
    )
    return DataLoader(dataset, **common)


In [31]:
def make_variants(
    square_rgb : np.ndarray,
    n_rotations : int,
    scales : List[int],
    gsd_drone : float,
    gsd_map : float,
) -> List[Tuple[np.ndarray, int, float]]:

    if isinstance(square_rgb, torch.Tensor):
        square_rgb = square_rgb.numpy()

    variants : List[Tuple[np.ndarray, int, float]] = []
    for scale in scales:
        size = max(64, int(square_rgb.shape[0] * gsd_drone * scale / gsd_map))
        resized_quare = cv2.resize(
            square_rgb,
            (size, size),
            interpolation=cv2.INTER_AREA,
        )
        for rot_idx in range(n_rotations):
            rotated = np.ascontiguousarray(np.rot90(resized_quare, rot_idx))
            variants.append((rotated, scale, rot_idx))

    return variants

In [32]:
def refine_query_variants(
    image: np.ndarray,
    query_scales: tuple[float, ...],
    n_rot: int,
    gsd_drone: float,
    gsd_map: float,
) -> list[tuple[np.ndarray, int, float]]:
    """Готує варіанти для локального матчингу без зміни пропорцій кадру."""
    height, width = image.shape[:2]
    variants: list[tuple[np.ndarray, int, float]] = []

    for scale in query_scales:
        map_width = max(64, int(width * gsd_drone * scale / gsd_map))
        map_height = max(64, int(height * gsd_drone * scale / gsd_map))
        scaled = cv2.resize(
            image,
            (map_width, map_height),
            interpolation=cv2.INTER_AREA,
        )

        for rotation_index in range(n_rot):
            rotated = np.ascontiguousarray(np.rot90(scaled, rotation_index))
            variants.append((rotated, rotation_index, scale))

    return variants

In [33]:
class ProcessedDroneDataset(Dataset):
    def __init__(self, dataset, n_rotations, scales, gsd_map, gsd_drone, dino_size=448):
        self.dataset = dataset
        self.n_rotations = n_rotations
        self.scales = scales
        self.gsd_map = gsd_map
        self.gsd_drone = gsd_drone

        # We need a fixed dimension to ensure batching works
        self.dino_size = dino_size

    def __getitem__(self, idx):
        # Load single image
        image_rgb, metadata, _ = self.dataset[idx]
        square = to_square(image_rgb)

        # variants is a list of tuples: (rotated_img, scale, rot_idx)
        variants = make_variants(square, self.n_rotations, self.scales, self.gsd_drone, self.gsd_map)

        # Resize each variant to dino_size before stacking
        processed_images = []
        for v in variants:
            img = v[0]
            # Resize to uniform shape so np.stack doesn't crash
            img_resized = cv2.resize(img, (self.dino_size, self.dino_size), interpolation=cv2.INTER_AREA)
            processed_images.append(img_resized)

        variant_images = np.stack(processed_images)

        # Store metadata about the variants (scale and rotation)
        variant_info = np.array([[v[1], v[2]] for v in variants])

        return variant_images, metadata, variant_info

    def __len__(self):
        return len(self.dataset)

In [34]:
def build_processed_drone_loader(dataset : ProcessedDroneDataset) -> DataLoader:
    common = dict(
        batch_size=config.dataset.batch_size,
        shuffle=config.dataset.shuffle,
        num_workers=config.dataset.num_workers,
        pin_memory=True if device.type == "cuda" else False,
    )
    return DataLoader(dataset, **common)

## Extraction of embeddings

In [35]:
# extract_embeddings # Get embds for map chunks
@torch.no_grad()
def extract_embeddings(
        model : torch.nn.Module,
        images_rgb: np.ndarray,
        agg : str,
        gem_p : int = 8
) -> torch.Tensor:
    batch_embeddings : List[torch.Tensor] = []

    images_rgb_torch = (torch.from_numpy(images_rgb).permute(0, 3, 1, 2).float().div_(255))

    images_rgb_torch = ((images_rgb_torch - IMNET_MEAN) / IMNET_STD).half().to(device)

    hidden_states = model(pixel_values=images_rgb_torch).last_hidden_state

    if agg == "cls":
        embeddings = hidden_states[:, 0]

    else:
        patch_tokens = hidden_states[:, 1:].float()

        if agg == "mean":
            embeddings = patch_tokens.mean(dim=1)

        elif agg == "gem":
            embeddings = (
                patch_tokens.clamp(min=1e-6).pow(gem_p).mean(dim=1).pow(1.0 / gem_p)
            )

        else:
            raise ValueError(f"Невідомий спосіб агрегації: {agg}")

        embeddings = F.normalize(
            embeddings.float(),
            dim=-1,
        ).cpu()
        batch_embeddings.append(embeddings)

    return torch.cat(batch_embeddings, dim=0)

In [36]:
def get_tiles_embeddings(
        model : torch.nn.Module,
        tiles_loader : DataLoader, # batch_size of tiles
        dino_size : int,
) -> Tuple[torch.Tensor, np.ndarray]:
    _, map_path = get_map()

    tile_size = tiles_loader.dataset.tile_size
    grid = np.asarray(tiles_loader.dataset.tiles)

    embeddings_batches : List[torch.Tensor] = []
    for tiles_batch in tqdm(tiles_loader, desc="Processing tiles embeddings"):
        with rasterio.open(map_path) as src:
            tiles_images = [
                (lambda x, y: np.ascontiguousarray(
                    src.read(
                        indexes=[1,2,3],
                        window=Window(x, y, tile_size, tile_size),
                    ).transpose(1,2,0)
                ))(x, y)
                for x, y in zip(tiles_batch[0], tiles_batch[1])
            ]
            resized_images_rgb = np.stack(
                [
                    cv2.resize(
                        image,
                        (dino_size, dino_size),
                        interpolation=cv2.INTER_AREA,
                    )
                    for image in tiles_images
                ]
            )
            batch_embeddings = extract_embeddings(model=model, images_rgb=resized_images_rgb, agg="gem")
            embeddings_batches.append(batch_embeddings)
    embeddings = torch.cat(embeddings_batches, dim=0)

    return embeddings, grid

In [37]:
tiles_dataset = TilesDataset(
    stride=config.tiles.stride,
    tile_size=config.tiles.tile_size,
)

images_loader = build_images_loader(dataset=images_dataset)
tiles_loader = build_tiles_loader(dataset=tiles_dataset)

tiles_embs, grid = get_tiles_embeddings(
    model=dinov2_model,
    tiles_loader=tiles_loader,
    dino_size=config.tiles.dino_size,
)


Processing tiles embeddings: 100%|██████████| 33/33 [03:42<00:00,  6.75s/it]


In [38]:
cv2.setNumThreads(0)
variants_common = dict(
    n_rotations=config.dataset.n_rotations,
    scales=config.dataset.scales,
    gsd_map=config.gsd_map,
    gsd_drone=config.gsd_drone,
)
processed_dataset = ProcessedDroneDataset(
    dataset=images_dataset,
    n_rotations=config.dataset.n_rotations,
    scales=config.dataset.scales,
    gsd_map=config.gsd_map,
    gsd_drone=config.gsd_drone,
    dino_size=config.tiles.dino_size  # Add this!
)
processed_loader = build_processed_drone_loader(processed_dataset)

In [39]:
def get_drone_embeddings(
        loader: DataLoader,
        model: torch.nn.Module,
        agg: str = "gem",
        max_chunk_size: int = 16 # Adjust this based on your GPU VRAM
) -> Tuple[torch.Tensor, List[List[Tuple[float, int]]]]:

    embeddings_batches = []
    variants_metadata = []

    model.eval()
    with torch.no_grad():
        with tqdm(total=len(loader), desc="Processing drone variants") as pbar:
            for images_batch, _, info_batch in loader:

                # images_batch shape: (B, N_variants, H, W, 3)
                B, N, H, W, C = images_batch.shape

                # Flatten batch and variants dimensions into a NumPy array
                images_flat = images_batch.view(-1, H, W, C).numpy()

                # Process in small chunks to prevent CUDA Out of Memory errors
                flat_embeddings = []
                for i in range(0, len(images_flat), max_chunk_size):
                    chunk = images_flat[i : i + max_chunk_size]
                    chunk_emb = extract_embeddings(model=model, images_rgb=chunk, agg=agg)
                    flat_embeddings.append(chunk_emb)

                # Concatenate chunks back together
                flat_embeddings = torch.cat(flat_embeddings, dim=0)

                # Reshape back to group by original batch: (B, N_variants, Embed_Dim)
                batch_embeddings = flat_embeddings.view(B, N, -1)

                embeddings_batches.append(batch_embeddings)
                variants_metadata.extend(info_batch.tolist())

                pbar.update(1)

    return torch.cat(embeddings_batches, dim=0), variants_metadata


In [40]:

drone_embs, variants_metadata = get_drone_embeddings(
    loader=processed_loader,
    model=dinov2_model,
    agg=config.dataset.agg,
)


Processing drone variants: 100%|██████████| 22/22 [13:40<00:00, 37.28s/it]


In [41]:
print(drone_embs.shape)

torch.Size([344, 12, 1536])


In [42]:
similarities = drone_embs @ tiles_embs.T

best_similarities, best_variant_indices = similarities.max(dim=1)

tile_order = best_similarities.argsort(dim=1, descending=True)

assert tile_order.shape == (len(images_dataset.images), len(grid))
assert best_variant_indices.shape == tile_order.shape
assert best_similarities.shape == tile_order.shape

In [43]:
# Retrieval-only: прогноз у центрі найсхожішого тайла.
top1_tile_indices = tile_order[:, 0]
top1_tiles = grid[top1_tile_indices]

pred_x = top1_tiles[:, 0] + config.tiles.tile_size / 2
pred_y = top1_tiles[:, 1] + config.tiles.tile_size / 2

pred_lat, pred_lon = xy_to_latlon(
    x=pred_x,
    y=pred_y,
)

filenames = images_dataset.images
evaluation_meta = images_dataset.metadata.set_index("filename").loc[filenames].reset_index()

retrieval_errors = mean_haversine_error(
    lat1=evaluation_meta["lat"].to_numpy(),
    lon1=evaluation_meta["lon"].to_numpy(),
    lat2=pred_lat,
    lon2=pred_lon,
)

frame_indices = np.arange(len(filenames))
top1_similarities = best_similarities[
    frame_indices,
    top1_tile_indices,
]

retrieval_only_df = pd.DataFrame(
    {
        "filename": filenames,
        "gt_lat": evaluation_meta["lat"].to_numpy(),
        "gt_lon": evaluation_meta["lon"].to_numpy(),
        "pred_lat": pred_lat,
        "pred_lon": pred_lon,
        "error_m": retrieval_errors,
        "top1_similarity": top1_similarities,
    }
)

print(f"Retrieval-only MHE: {retrieval_errors.mean():.1f} м")
print(f"Median: {np.median(retrieval_errors):.1f} м")
print(f"P90: {np.percentile(retrieval_errors, 90):.1f} м")
print(f"≤50 м:  {100 * np.mean(retrieval_errors <= 50):.1f}%")
print(f"≤100 м: {100 * np.mean(retrieval_errors <= 100):.1f}%")
print(f"≤500 м: {100 * np.mean(retrieval_errors <= 500):.1f}%")

retrieval_only_df.head()

Retrieval-only MHE: 489.3 м
Median: 256.0 м
P90: 1348.0 м
≤50 м:  14.5%
≤100 м: 31.1%
≤500 м: 68.0%


,filename,gt_lat,gt_lon,pred_lat,pred_lon,error_m,top1_similarity
0,06_0001.JPG,32.351541,109.646761,32.351201,109.646149,68.841486,0.982569
1,06_0002.JPG,32.351025,109.646945,32.353948,109.644775,383.583922,0.980382
2,06_0003.JPG,32.350498,109.647117,32.351201,109.646149,119.911762,0.978451
3,06_0004.JPG,32.349982,109.647289,32.348317,109.639282,774.517702,0.978813
4,06_0005.JPG,32.349461,109.647460,32.350514,109.646149,170.014052,0.975877


## Keypoints & Matches

In [44]:
if isinstance(tile_order, torch.Tensor):
    tile_order = tile_order.numpy()
    best_variant_indices = best_variant_indices.numpy()
    best_similarities = best_similarities.numpy()

# Ембедінги видаляються перед локальним матчингом, щоб звільнити GPU-пам'ять.
del dinov2_model, drone_embs, tiles_embs, similarities
torch.cuda.empty_cache()

In [45]:
class SPLGModel():
    """SuperPoint + LightGlue model."""
    
    def __init__(self, tile_size, num_keypoints : int = 2048):
        self.extractor = SuperPoint(num_keypoints=num_keypoints).eval().to(device)
        self.matcher = LightGlue(features="superpoint").eval().to(device)
        
        self.tile_size = tile_size
        
        _, self.map_width, self.map_height = prepare_map_data()


    def __call__(self, 
        frame_variants: list[tuple[np.ndarray, int, float]],
        frame_tile_order: np.ndarray,
        frame_best_variant_indices: np.ndarray,
        grid: np.ndarray,
        topk: int,
        min_inliers: int,
        good_inliers: int,
        rotation_retry_ranks: int,
        rotation_retry_min_inliers: int,
        magsac_threshold: float,
    ):
        """Шукає найкращий геометрично правдоподібний матч серед retrieval-кандидатів."""
        query_feature_cache: dict[int, dict] = {}
        best_match: dict | None = None
    
        for rank in range(topk):
            tile_index = int(frame_tile_order[rank])
            x0, y0 = (int(value) for value in grid[tile_index])
            primary_variant_index = int(frame_best_variant_indices[tile_index])
    
            tile_image = read_map_window(x0=x0, y0=y0, size=self.tile_size)
            with torch.no_grad():
                tile_features = self.extractor.extract(
                    numpy_image_to_torch(tile_image).to(device)
                )
    
            variant_indices = [primary_variant_index]
            variant_position = 0
    
            while variant_position < len(variant_indices):
                variant_index = variant_indices[variant_position]
                query_image, rotation_index, query_scale = frame_variants[variant_index]
    
                if variant_index not in query_feature_cache:
                    with torch.no_grad():
                        query_feature_cache[variant_index] = self.extractor.extract(
                            numpy_image_to_torch(query_image).to(device)
                        )
    
                homography, n_inliers, n_matches = self.match_feature_pair(
                    query_features=query_feature_cache[variant_index],
                    tile_features=tile_features,
                    magsac_threshold=magsac_threshold,
                )
    
                # Для перших кандидатів інші ротації того самого масштабу перевіряються
                # лише тоді, коли початковий матч близький до порогу прийняття.
                if (
                    variant_position == 0
                    and rank < rotation_retry_ranks
                    and rotation_retry_min_inliers <= n_inliers < min_inliers
                ):
                    variant_indices.extend(
                        index
                        for index, (_, _, scale) in enumerate(frame_variants)
                        if scale == query_scale and index != primary_variant_index
                    )
    
                if homography is not None:
                    query_height, query_width = query_image.shape[:2]
                    query_center = np.array(
                        [[[query_width / 2, query_height / 2]]], dtype=np.float64
                    )
                    tile_center = cv2.perspectiveTransform(query_center, homography)[0, 0]
    
                    margin = self.tile_size * 0.5
                    pred_x = x0 + tile_center[0]
                    pred_y = y0 + tile_center[1]
                    plausible = (
                        np.isfinite(tile_center).all()
                        and -margin <= tile_center[0] <= self.tile_size + margin
                        and -margin <= tile_center[1] <= self.tile_size + margin
                        and 0 <= pred_x < self.map_width
                        and 0 <= pred_y < self.map_height
                    )
    
                    if plausible and (
                        best_match is None or n_inliers > best_match["n_inliers"]
                    ):
                        best_match = {
                            "homography": homography,
                            "tile_center": tile_center,
                            "n_inliers": n_inliers,
                            "n_matches": n_matches,
                            "inlier_ratio": n_inliers / max(n_matches, 1),
                            "rank": rank + 1,
                            "tile_index": tile_index,
                            "x0": x0,
                            "y0": y0,
                            "query_shape": query_image.shape[:2],
                            "variant_index": variant_index,
                            "rotation_index": rotation_index,
                            "query_scale": query_scale,
                            "rotation_retry": variant_index != primary_variant_index,
                        }
    
                if best_match is not None and best_match["n_inliers"] >= good_inliers:
                    break
    
                variant_position += 1
    
            if best_match is not None and best_match["n_inliers"] >= good_inliers:
                break
    
        return best_match
    
    
    def match_feature_pair(self, query_features: dict, tile_features: dict, magsac_threshold: float) \
            -> tuple[np.ndarray | None, int, int]:

        with torch.no_grad():
            match_output = self.matcher(
                {
                    "image0": query_features,
                    "image1": tile_features,
                }
            )
    
        matches = rbd(match_output)["matches"].cpu().numpy()

        if len(matches) < 8:
            return None, 0, len(matches)
    
        query_keypoints = rbd(query_features)["keypoints"].cpu().numpy()
        tile_keypoints = rbd(tile_features)["keypoints"].cpu().numpy()
    
        matched_query_points = query_keypoints[matches[:, 0]]
        matched_tile_points = tile_keypoints[matches[:, 1]]
    
        homography, inlier_mask = cv2.findHomography(
            matched_query_points,
            matched_tile_points,
            cv2.USAC_MAGSAC,
            magsac_threshold,
            maxIters=10_000,
            confidence=0.9999,
        )
    
        if homography is None or inlier_mask is None:
            return None, 0, len(matches)
    
        n_inliers = int(inlier_mask.sum())
        return homography, n_inliers, len(matches)



In [46]:
SPLG_model = SPLGModel(
    tile_size=config.tiles.tile_size,
    num_keypoints=config.model.num_keypoints
)

Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/superpoint_v1.pth" to /root/.cache/torch/hub/checkpoints/superpoint_v1.pth


100%|██████████| 4.96M/4.96M [00:00<00:00, 103MB/s]


Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/superpoint_lightglue.pth" to /root/.cache/torch/hub/checkpoints/superpoint_lightglue_v0-1_arxiv.pth


100%|██████████| 45.3M/45.3M [00:00<00:00, 246MB/s]


In [47]:
def weighted_tile_centroid(
    tile_order: np.ndarray,
    tile_similarities: np.ndarray,
    grid: np.ndarray,
    tile_size: int,
    topk: int,
    temperature: float,
) -> tuple[float, float]:
    """Обчислює зважений центр Top-K retrieval-тайлів.

    Args:
        tile_order: індекси тайлів від найкращого до найгіршого.
        tile_similarities: similarity поточного кадру з усіма тайлами.
        grid: координати лівих верхніх кутів усіх тайлів.
        tile_size: сторона тайла у пікселях.
        topk: кількість retrieval-кандидатів для fallback.
        temperature: температура softmax-зважування.

    Returns:
        координати (x, y) зваженого центра у пікселях карти.
    """
    candidate_indices = tile_order[:topk]
    candidate_scores = tile_similarities[candidate_indices].astype(np.float64)

    # Віднімання максимуму не змінює softmax-ваги,
    # але захищає exp від числового переповнення.
    weights = np.exp((candidate_scores - candidate_scores.max()) / temperature)
    weights = weights / weights.sum()

    # Координати grid задають лівий верхній кут,
    # тому додається половина сторони тайла.
    candidate_centers_x = grid[candidate_indices, 0] + tile_size / 2
    candidate_centers_y = grid[candidate_indices, 1] + tile_size / 2

    pred_x = float(np.sum(weights * candidate_centers_x))
    pred_y = float(np.sum(weights * candidate_centers_y))

    return pred_x, pred_y

In [50]:
prediction_rows: list[dict] = []
refine_start_time = time.time()

_, map_width, map_height = prepare_map_data()

for frame_index, filename in enumerate(tqdm(filenames, desc="Refine")):
    image_rgb, _, _ = images_dataset.find_by_name(filename)
    # Масштаби й ротації мають той самий порядок, що й у retrieval, але
    # пропорції початкового кадру для локального матчингу не змінюються.
    frame_variants = refine_query_variants(
        image=image_rgb,
        query_scales=config.dataset.scales,
        n_rot=config.dataset.n_rotations,
        gsd_drone=config.gsd_drone,
        gsd_map=config.gsd_map,
    )

    best_match = SPLG_model(
        frame_variants=frame_variants,
        frame_tile_order=tile_order[frame_index],
        frame_best_variant_indices=best_variant_indices[frame_index],
        grid=grid,
        topk=config.model.top_k,
        min_inliers=config.inliers.min,
        good_inliers=config.inliers.good,
        rotation_retry_ranks=config.rotation_retry_ranks,
        rotation_retry_min_inliers=config.inliers.rotation_retry_min,
        magsac_threshold=config.model.magsac_thresh,
    )

    selected_tile_index = -1
    selected_tile_x0 = np.nan
    selected_tile_y0 = np.nan
    tile_center_x = np.nan
    tile_center_y = np.nan
    variant_index = -1
    rotation_deg = np.nan
    query_scale = np.nan
    n_matches = 0
    inlier_ratio = 0.0
    rotation_retry = False

    if best_match is not None and best_match["n_inliers"] >= config.inliers.min:
        tile_center = best_match["tile_center"]
        pred_x = best_match["x0"] + tile_center[0]
        pred_y = best_match["y0"] + tile_center[1]

        method = "lightglue"
        n_inliers = best_match["n_inliers"]
        chosen_rank = best_match["rank"]
        selected_tile_index = best_match["tile_index"]
        selected_tile_x0 = best_match["x0"]
        selected_tile_y0 = best_match["y0"]
        tile_center_x = float(tile_center[0])
        tile_center_y = float(tile_center[1])
        variant_index = best_match["variant_index"]
        rotation_deg = best_match["rotation_index"] * 90
        query_scale = best_match["query_scale"]
        n_matches = best_match["n_matches"]
        inlier_ratio = best_match["inlier_ratio"]
        rotation_retry = best_match["rotation_retry"]

    else:
        # Fallback використовує п'ять найсильніших із 20 retrieval-кандидатів.
        pred_x, pred_y = weighted_tile_centroid(
            tile_order=tile_order[frame_index],
            tile_similarities=best_similarities[frame_index],
            grid=grid,
            tile_size=config.tiles.tile_size,
            topk=config.fallback.top_k,
            temperature=config.fallback.temp,
        )

        method = "fallback_wcentroid"
        n_inliers = 0
        chosen_rank = 0

    pred_lat, pred_lon = xy_to_latlon(
        x=pred_x,
        y=pred_y,
    )

    top1_tile_index = int(tile_order[frame_index, 0])
    top1_similarity = best_similarities[
        frame_index,
        top1_tile_index,
    ]

    prediction_rows.append(
        {
            "filename": filename,
            "pred_lat": float(pred_lat),
            "pred_lon": float(pred_lon),
            "pred_x": float(pred_x),
            "pred_y": float(pred_y),
            "top1_similarity": float(top1_similarity),
            "method": method,
            "n_inliers": int(n_inliers),
            "chosen_rank": int(chosen_rank),
            "selected_tile_index": int(selected_tile_index),
            "selected_tile_x0": selected_tile_x0,
            "selected_tile_y0": selected_tile_y0,
            "tile_center_x": tile_center_x,
            "tile_center_y": tile_center_y,
            "variant_index": int(variant_index),
            "rotation_deg": rotation_deg,
            "query_scale": query_scale,
            "n_matches": int(n_matches),
            "inlier_ratio": float(inlier_ratio),
            "rotation_retry": bool(rotation_retry),
        }
    )

predictions_df = pd.DataFrame(prediction_rows)

refine_runtime_minutes = (time.time() - refine_start_time) / 60

assert len(predictions_df) == len(filenames)
assert predictions_df[["pred_lat", "pred_lon"]].notna().all().all()
assert predictions_df["pred_x"].between(0, map_width, inclusive="left").all()
assert predictions_df["pred_y"].between(0, map_height, inclusive="left").all()

print(f"Refine завершено за {refine_runtime_minutes:.1f} хв")
print(predictions_df["method"].value_counts().to_string())

predictions_df.head()

Refine: 100%|██████████| 344/344 [10:43<00:00,  1.87s/it]

Refine завершено за 10.7 хв
method
lightglue             207
fallback_wcentroid    137


,filename,pred_lat,pred_lon,pred_x,pred_y,top1_similarity,method,n_inliers,chosen_rank,selected_tile_index,selected_tile_x0,selected_tile_y0,tile_center_x,tile_center_y,variant_index,rotation_deg,query_scale,n_matches,inlier_ratio,rotation_retry
0,06_0001.JPG,32.354605,109.643969,3284.361155,6923.704108,0.982569,fallback_wcentroid,0,0,-1,NaN,NaN,NaN,NaN,-1,NaN,NaN,0,0.0,False
1,06_0002.JPG,32.351049,109.644373,3434.988092,8249.491990,0.980382,fallback_wcentroid,0,0,-1,NaN,NaN,NaN,NaN,-1,NaN,NaN,0,0.0,False
2,06_0003.JPG,32.350486,109.643131,2972.034696,8459.568965,0.978451,fallback_wcentroid,0,0,-1,NaN,NaN,NaN,NaN,-1,NaN,NaN,0,0.0,False
3,06_0004.JPG,32.350207,109.641884,2506.994557,8563.558966,0.978813,fallback_wcentroid,0,0,-1,NaN,NaN,NaN,NaN,-1,NaN,NaN,0,0.0,False
4,06_0005.JPG,32.351447,109.641353,2309.100420,8101.303937,0.975877,fallback_wcentroid,0,0,-1,NaN,NaN,NaN,NaN,-1,NaN,NaN,0,0.0,False


In [54]:
region, map_path = get_map()

# GT вирівнюється за точним порядком predictions_df.
# Жодне значення GT не використовувалося під час retrieval або refine.
ground_truth = images_dataset.metadata.set_index("filename").loc[predictions_df["filename"]].reset_index()

results_df = predictions_df.copy()
results_df.insert(
    1,
    "gt_lat",
    ground_truth["lat"].to_numpy(),
)
results_df.insert(
    2,
    "gt_lon",
    ground_truth["lon"].to_numpy(),
)

# Per-image Haversine error для всіх кадрів регіону,
# включно з позамапними та проблемними GT.
results_df["error_m"] = mean_haversine_error(
    lat1=results_df["gt_lat"].to_numpy(),
    lon1=results_df["gt_lon"].to_numpy(),
    lat2=results_df["pred_lat"].to_numpy(),
    lon2=results_df["pred_lon"].to_numpy(),
)

errors = results_df["error_m"].to_numpy()

# Recall@K: чи містить хоча б один із K retrieval-тайлів GT-точку.
gt_x, gt_y = latlon_to_xy(
    ground_truth["lat"].to_numpy(),
    ground_truth["lon"].to_numpy(),
)
retrieval_recall = {}
for recall_k in (1, 5, 10, 20):
    candidate_origins = grid[tile_order[:, :recall_k]]
    gt_hits = (
        (candidate_origins[:, :, 0] <= gt_x[:, None])
        & (gt_x[:, None] < candidate_origins[:, :, 0] + config.tiles.tile_size)
        & (candidate_origins[:, :, 1] <= gt_y[:, None])
        & (gt_y[:, None] < candidate_origins[:, :, 1] + config.tiles.tile_size)
    ).any(axis=1)
    retrieval_recall[recall_k] = 100 * gt_hits.mean()

headline_metrics = {
    "N": len(results_df),
    "MHE, м": errors.mean(),
    "Median, м": np.median(errors),
    "P90, м": np.percentile(errors, 90),
    "≤50 м, %": 100 * np.mean(errors <= 50),
    "≤100 м, %": 100 * np.mean(errors <= 100),
    "≤500 м, %": 100 * np.mean(errors <= 500),
    "Recall@1, %": retrieval_recall[1],
    "Recall@5, %": retrieval_recall[5],
    "Recall@10, %": retrieval_recall[10],
    "Recall@20, %": retrieval_recall[20],
    "Refined, %": 100 * np.mean(results_df["method"] == "lightglue"),
}

metrics_table = pd.DataFrame(
    {
        "Метрика": headline_metrics.keys(),
        "Значення": headline_metrics.values(),
    }
)

metrics_table["Значення"] = metrics_table["Значення"].round(1)

predictions_path = Path(f"predictions_{region}.csv")
results_df.to_csv(predictions_path, index=False)
print(f"Прогнози збережено: {predictions_path.resolve()}")

metrics_table

Прогнози збережено: /content/predictions_06.csv


,Метрика,Значення
0,N,344.0
1,"MHE, м",300.4
2,"Median, м",107.0
3,"P90, м",824.6
4,"≤50 м, %",46.2
5,"≤100 м, %",49.4
6,"≤500 м, %",81.4
7,"Recall@1, %",43.0
8,"Recall@5, %",59.9
9,"Recall@10, %",65.4
